# System research notebook

Copied verbatim from `research/systems/_template/template.ipynb` by `quantlab.systems.create_system` (DESIGN §1/§3/§10 Phase 2). Every stage cell below is a thin wrapper over `quantlab`; heavy logic lives in the library, not here. All of it reads the `SYSTEM`/`CARD` dicts built in §0 from this folder's own `hypothesis.md` -- nothing below is specific to one system, including this heading; the system's name/slug is shown once §0 runs.

## 0. Setup

In [ ]:
from pathlib import Path

from quantlab import config
from quantlab.systems import read_card

# Jupyter's default working directory is the notebook's own folder; nbconvert callers should
# pass resources={"metadata": {"path": <this folder>}} to ExecutePreprocessor if that isn't
# already true. hypothesis.md lives right next to this notebook.
SYSTEM_DIR = Path.cwd()
CARD = read_card(SYSTEM_DIR / "hypothesis.md")
SYSTEM = {
    "book": CARD["book"],
    "issue": CARD["issue"],
    "slug": CARD["slug"],
    "name": CARD.get("name", CARD["slug"]),
    "risk_type": CARD.get("risk_type"),
    "status": CARD.get("status", "testing"),
    # Set by strategy-engineer once the instrument/timeframe is picked (S3): add `symbol:` /
    # `timeframe:` (and optionally `start:` / `end:`) to hypothesis.md's front matter.
    "symbol": CARD.get("symbol"),
    "timeframe": CARD.get("timeframe"),
    "start": CARD.get("start"),
    "end": CARD.get("end"),
}
LEDGER_DIR = config.LEDGER_DIR
STUDIES_DIR = config.STUDIES_DIR
RESULTS_DIR = SYSTEM_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SYSTEM

## 1. Hypothesis

In [ ]:
from IPython.display import Markdown, display

display(Markdown(f"**Status:** {SYSTEM['status']}  ·  **Risk type:** {SYSTEM['risk_type']}  ·  "
                 f"**Issue:** #{SYSTEM['issue']}"))
display(Markdown(CARD.get("body", "")))

## 2. Implementation

In [ ]:
import inspect

from quantlab.costs import InstrumentSpec
from quantlab.strategies import get_strategy
from quantlab.testing import (
    assert_engine_causal,
    assert_no_lookahead,
    assert_strategy_source_clean,
    synthetic_bars,
)

StrategyCls = get_strategy(SYSTEM["slug"])
assert_strategy_source_clean(inspect.getfile(StrategyCls))

# Same class/Params resolution order as quantlab.evaluators.RuleEvaluator.build_strategy: a
# strategy is built from its declared prior (default) parameters here, so this audit exercises
# the real class with the card's own prior values, not an arbitrary stand-in.
_params_cls = getattr(StrategyCls, "params_cls", None) or getattr(StrategyCls, "Params", None)
_factory = (lambda: StrategyCls(_params_cls())) if _params_cls is not None else StrategyCls

# Generic, book/timeframe-agnostic bars/spec for the audit -- the real per-system test module
# (tests/, written alongside this notebook at S2) additionally runs assert_engine_causal on real
# dev data with m1=/timeframe= for the system's actual instrument.
_audit_bars = synthetic_bars(600, seed=0, timeframe="H1")
assert_no_lookahead(_factory, _audit_bars)

_audit_spec = InstrumentSpec(
    symbol="AUDIT", digits=5, point=1e-5, contract_size=100_000.0, tick_size=1e-5,
    volume_min=0.01, volume_step=0.01, volume_max=100.0, base_ccy="EUR", quote_ccy="USD",
    swap_mode="points", swap_long=0.0, swap_short=0.0, swap_3day=2,
    commission_per_lot_rt=0.0, stops_level=0.0, calibrated=True,
)
assert_engine_causal(_factory(), _audit_bars, _audit_spec, n_checks=8, seed=1, min_history=60)

print(f"{StrategyCls.name}: look-ahead audit clean (risk_type={StrategyCls.risk_type})")

## 3. Baseline

In [ ]:
if SYSTEM.get("symbol") and SYSTEM.get("timeframe"):
    from quantlab.evaluators import RuleEvaluator

    evaluator = RuleEvaluator(
        StrategyCls, symbol=SYSTEM["symbol"], timeframe=SYSTEM["timeframe"], book=SYSTEM["book"],
        start=SYSTEM.get("start"), end=SYSTEM.get("end"),
    )
    baseline = evaluator({})   # {} = the card's prior (default) parameters, costs on
    print(evaluator.describe())
    baseline.metrics
else:
    evaluator = None
    baseline = None
    print("S3 pending: add `symbol:` / `timeframe:` to hypothesis.md's front matter, then re-run.")

## 4. Optimisation

In [ ]:
if evaluator is not None:
    from quantlab import opt

    # S4 (optimization-architect): declare the real search space here -- every numeric Param
    # needs plateau_scale="relative" or plateau_step=... (DESIGN §4.2, pre-registered on the
    # card) -- then run_study(...). Left unset in the template so it never launches a study with
    # placeholder bounds.
    space = None   # opt.SearchSpace((opt.IntParam(...), ...))
    study = None
    if space is not None:
        study = opt.run_study(
            evaluator, space, book=SYSTEM["book"], system=SYSTEM["slug"], issue=SYSTEM["issue"], attempt=1,
        )
else:
    space = None
    study = None
    print("S4 pending: needs a baseline (S3) first.")

## 5. Validation

In [ ]:
if study is not None:
    import polars as pl

    from quantlab import gates

    gate_report = gates.evaluate_gates(study, evaluator, periods_per_year=evaluator.periods_per_year, log=True)
    gate_table = pl.DataFrame([
        {"gate": r.gate, "value": r.display if r.display is not None else r.value,
         "threshold": r.threshold, "status": r.status, "interpretation": r.interpretation}
        for r in gate_report.rows
    ])
    gate_table
else:
    gate_report = None
    gate_table = None
    print("S5 pending: needs an optimisation study (S4) first.")

## 6. Red-team findings

In [ ]:
FINDINGS_PATH = RESULTS_DIR / "redteam_findings.md"
if FINDINGS_PATH.exists():
    from IPython.display import Markdown, display

    display(Markdown(FINDINGS_PATH.read_text()))
else:
    print(f"S6 pending: no findings file yet at {FINDINGS_PATH}")

## 7. Report

In [ ]:
if study is not None and gate_report is not None:
    from quantlab import report

    ts = report.tear_sheet(study.study_id, evaluator=evaluator, risk_type=SYSTEM.get("risk_type"))
    ts.write_results(RESULTS_DIR)
    report.write_card(
        ts, slug=SYSTEM["slug"], name=SYSTEM["name"], idea=CARD.get("idea", ""),
        status=SYSTEM.get("status", "testing"), issue=SYSTEM["issue"], book=SYSTEM["book"],
    )
    ts.headline
else:
    ts = None
    print("S7 pending: needs validation (S5) first.")

## 8. Holdout

In [ ]:
# Status only. The holdout is unlocked once, by the user, via `/unlock-holdout <system>` -- never
# from this notebook. This cell never calls quantlab.data with include_holdout=True.
print(f"S8 holdout: system status = {SYSTEM.get('status')!r} (book {SYSTEM['book']}). "
     "Run `/unlock-holdout` yourself when checkpoint B is reached (DESIGN §3, §4.4).")